In [5]:
!pip install -q mediapipe==0.10.14

Подключаем необходимые библиотеки

In [6]:
import os # Возможно не нужен будет
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import cv2 # Работа с видео (и картинками)
from pathlib import Path # Работа с путями к файлам
import mediapipe as mp
import matplotlib.pyplot as plt

Конфиги

In [8]:
ANNOTATIONS_PATH = "/kaggle/input/datasets/kapitanov/slovo/annotations.csv"
RUSSIAN_LETTERS = list("АБВГДЕЁЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯ")

VIDEOS_DIR = Path("/kaggle/input/datasets/kapitanov/slovo")
OUT_DIR = Path("/kaggle/working/dataset")
FRAMES_PER_VIDEO = 10        
hands = mp.solutions.hands.Hands(static_image_mode=True, max_num_hands=1, min_detection_confidence=0.5)

W0000 00:00:1782999186.403086     161 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1782999186.442971     161 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Подключаем библиотеку и выделяем только необходимые признаки, а именно дактили

In [13]:
df = pd.read_csv(ANNOTATIONS_PATH, sep="\t")
print(df.columns.tolist())
print(df.head())

dactyl = df[df["text"].isin(RUSSIAN_LETTERS)]
print("Уникальных букв осталось:", dactyl["text"].nunique())
print(dactyl["text"].value_counts())     # сколько видео на каждую букву

['attachment_id', 'text', 'user_id', 'height', 'width', 'length', 'train', 'begin', 'end']
                          attachment_id text  \
0  44e8d2a0-7e01-450b-90b0-beb7400d2c1e    Ё   
1  df5b08f0-41d1-4572-889c-8b893e71069b    А   
2  17f53df4-c467-4aff-9f48-20687b63d49a    Р   
3  e3add916-c708-4339-ad98-7e2740be29e9    Е   
4  bd7272ed-1850-48f1-a2a8-c8fed523dc37    Ч   

                            user_id  height  width  length  train  begin  end  
0  185bd3a81d9d618518d10abebf0d17a8    1920   1080   156.0   True     36  112  
1  185bd3a81d9d618518d10abebf0d17a8    1920   1080   150.0   True     36   76  
2  185bd3a81d9d618518d10abebf0d17a8    1920   1080   133.0   True     40   97  
3  185bd3a81d9d618518d10abebf0d17a8    1920   1080   144.0   True     43  107  
4  185bd3a81d9d618518d10abebf0d17a8    1920   1080    96.0   True     20   70  
Уникальных букв осталось: 33
text
Ё    20
А    20
Р    20
Е    20
Ч    20
Л    20
Ц    20
С    20
Й    20
З    20
Ь    20
Я    20
Б    20
Щ 

Функции
cut_frames - возвращает список кадров конкретного видео, которое подается на вход функции, а также начало и конца содержимого видео

hand_box_yolo - возвращает рамки вокруг ладони (дактиля)

In [ ]:
def cut_frames(video_path, begin, end, n=FRAMES_PER_VIDEO):
    cap = cv2.VideoCapture(str(video_path))
    step = (end - begin) / (n + 1)
    frame_ids = set(int(begin + step * k) for k in range(1, n + 1))
    frames = []
    index = 0
    while cap.isOpened():
        ok, frame = cap.read()
        if not ok or index > end:
            break
        if index in frame_ids:
            frames.append(frame)
        index += 1
    cap.release()                             
    return frames

def hand_box_yolo(frame, margin=0.10):
    res = hands.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    if not res.multi_hand_landmarks: 
        return None
    pts = res.multi_hand_landmarks[0].landmark
    xs = [p.x for p in pts] 
    ys = [p.y for p in pts]
    x1, x2 = max(0, min(xs)-margin), min(1, max(xs)+margin)
    y1, y2 = max(0, min(ys)-margin), min(1, max(ys)+margin)
    return (x1+x2)/2, (y1+y2)/2, x2-x1, y2-y1

video_paths = {p.stem: p for p in VIDEOS_DIR.rglob("*.mp4")}

In [ ]:
classes = sorted(dactyl["text"].unique())    
cls_id  = {c: i for i, c in enumerate(classes)}

for _, row in dactyl.iterrows():                   
    video = video_paths.get(row["attachment_id"])
    if video is None: continue
    split = "train" if row["train"] else "val"
    (OUT_DIR/"images"/split).mkdir(parents=True, exist_ok=True)
    (OUT_DIR/"labels"/split).mkdir(parents=True, exist_ok=True)

    for i, frame in enumerate(cut_frames(video, row["begin"], row["end"])):
        box = hand_box_yolo(frame)
        if box is None: 
            continue                    # рука не найдена — пропускаем кадр
        stem = f'{row["attachment_id"]}_{i}'
        cv2.imwrite(str(OUT_DIR/"images"/split/f"{stem}.jpg"), frame)
        cx, cy, w, h = box
        (OUT_DIR/"labels"/split/f"{stem}.txt").write_text(
            f"{cls_id[row['text']]} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n")

print("Готово. Классов:", len(classes))

In [ ]:
# сколько картинок и меток реально получилось?
for split in ["train", "val"]:
    n_img = len(list((OUT_DIR/"images"/split).glob("*.jpg")))
    n_lbl = len(list((OUT_DIR/"labels"/split).glob("*.txt")))
    print(f"{split}: картинок {n_img}, меток {n_lbl}")

Реалзация Data-yaml - инструкция для обучения модели YOLO11v

In [14]:
import yaml

classes = sorted(dactyl["text"].unique()) 

data_config = {
    "path": str(OUT_DIR),          # корневая папка датасета
    "train": "images/train",       # где картинки для обучения (относительно path)
    "val": "images/val",           # где картинки для проверки
    "names": {i: c for i, c in enumerate(classes)}   # номер класса -> буква: {0:'А', 1:'Б', ...}
}

yaml_path = OUT_DIR / "data.yaml"
with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(data_config, f, allow_unicode=True)   # allow_unicode — чтобы буквы были русскими, а не кодами

print(open(yaml_path, encoding="utf-8").read())          # посмотрим, что записалось

names:
  0: Ё
  1: А
  2: Б
  3: В
  4: Г
  5: Д
  6: Е
  7: Ж
  8: З
  9: И
  10: Й
  11: К
  12: Л
  13: М
  14: Н
  15: О
  16: П
  17: Р
  18: С
  19: Т
  20: У
  21: Ф
  22: Х
  23: Ц
  24: Ч
  25: Ш
  26: Щ
  27: Ъ
  28: Ы
  29: Ь
  30: Э
  31: Ю
  32: Я
path: /kaggle/working/dataset
train: images/train
val: images/val



In [ ]:
for split in ["train", "val"]:
    n_img = len(list((OUT_DIR/"images"/split).glob("*.jpg")))
    n_lbl = len(list((OUT_DIR/"labels"/split).glob("*.txt")))
    print(f"{split}: картинок {n_img}, меток {n_lbl}")